# RQ2: Which features are most important for predicting financial fraud?

**Research Question:** What are the most influential features driving fraud detection using feature importance from tree-based models and correlation analysis?

**Hypothesis:** Behavioral features (velocity_score, spending_deviation_score, geo_anomaly_score) will rank higher than transaction attributes.

**Target Variable:** `is_fraud`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
import warnings, os
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.inspection import permutation_importance

print('Libraries loaded.')

In [ ]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
DATA_PATH = '/kaggle/input/financial-transactions-dataset-for-fraud-detection/financial_fraud_detection_dataset.csv'
df = pd.read_csv(DATA_PATH, nrows=50000)
print('Shape:', df.shape)

In [ ]:
TARGET = 'is_fraud'
DROP_COLS = ['transaction_id','timestamp','sender_account','receiver_account',
             'fraud_type','ip_address','device_hash']
df.drop(columns=[c for c in DROP_COLS if c in df.columns], inplace=True)

le = LabelEncoder()
for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col].astype(str))

df.dropna(inplace=True)
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print('Split done. Features:', X.shape[1])

In [ ]:
# Random Forest feature importance
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

fi_rf = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

# Gradient Boosting feature importance
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_train, y_train)
fi_gb = pd.Series(gb.feature_importances_, index=X.columns).sort_values(ascending=False)

df_fi = pd.DataFrame({
    'Feature': fi_rf.index,
    'RF_Importance': fi_rf.values,
    'GB_Importance': fi_gb[fi_rf.index].values
})
df_fi['Avg_Importance'] = (df_fi['RF_Importance'] + df_fi['GB_Importance']) / 2
df_fi = df_fi.sort_values('Avg_Importance', ascending=False).reset_index(drop=True)
print(df_fi.head(10))

In [ ]:
df_fi.to_csv('table_rq2_feature_importance.csv', index=False)
print('Table saved.')

In [ ]:
top_n = 10
df_top = df_fi.head(top_n)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: grouped bar
x = np.arange(top_n)
w = 0.35
axes[0].barh(x + w/2, df_top['RF_Importance'], w, label='Random Forest', color='#2196F3', alpha=0.85)
axes[0].barh(x - w/2, df_top['GB_Importance'], w, label='Gradient Boosting', color='#FF9800', alpha=0.85)
axes[0].set_yticks(x)
axes[0].set_yticklabels(df_top['Feature'], fontsize=10)
axes[0].set_xlabel('Importance Score', fontsize=12)
axes[0].set_title('Feature Importance:\nRF vs Gradient Boosting', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].xaxis.grid(True, linestyle='--', alpha=0.6)

# Right: correlation heatmap of top features
top_feats = df_top['Feature'].tolist() + [TARGET]
corr = df[top_feats].corr()
im = axes[1].imshow(corr, cmap='RdYlGn', vmin=-1, vmax=1, aspect='auto')
axes[1].set_xticks(range(len(top_feats)))
axes[1].set_yticks(range(len(top_feats)))
axes[1].set_xticklabels(top_feats, rotation=45, ha='right', fontsize=8)
axes[1].set_yticklabels(top_feats, fontsize=8)
plt.colorbar(im, ax=axes[1])
axes[1].set_title('Correlation Heatmap\n(Top Features + Target)', fontsize=12, fontweight='bold')
for i in range(len(top_feats)):
    for j in range(len(top_feats)):
        axes[1].text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center', fontsize=6)

plt.suptitle('RQ2: Feature Importance Analysis for Fraud Detection', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('figure_rq2_feature_importance.pdf', bbox_inches='tight', dpi=300)
plt.show()
print('Figure saved.')